<h1>Data Access and Utilization<span class="tocSkip"></span></h1>

# Objective:

- Get used to download data from [CMEMS](https://marine.copernicus.eu/) service
- Download and use
    - Time Series Dataset
    - Gridded product

# Data access

## CMEMS download tool
We can access data from CMEMS service using 2 methods
- [Web portal](https://data.marine.copernicus.eu/products)
- [Copernicus marine toolbox](https://help.marine.copernicus.eu/en/collections/9080063-copernicus-marine-toolbox)

The [python package](https://pypi.org/project/copernicusmarine/) of Copernicus Marine Toolbox is useful for multiple data downloading. However, this package is still in development phase and changes each year.

In this exercise we will use the [web portal](https://data.marine.copernicus.eu/products) to download the data we need.

## Data download
### Time series data
We will download area averaged sea level anomaly time series
1. [Global](https://data.marine.copernicus.eu/product/OMI_CLIMATE_SL_GLOBAL_area_averaged_anomalies/description)
2. Regional

    a. [Europe](https://data.marine.copernicus.eu/product/OMI_CLIMATE_SL_EUROPE_area_averaged_anomalies/description) (Northeast Atlantic Ocean and adjacent seas)
   
    b. [Baltic sea](https://data.marine.copernicus.eu/product/OMI_CLIMATE_SL_BALTIC_area_averaged_anomalies/description)
   
    c. [Black sea](https://data.marine.copernicus.eu/product/OMI_CLIMATE_SL_BLKSEA_area_averaged_anomalies/description)
   
    d. [Atlantic Iberian Biscay](https://data.marine.copernicus.eu/product/OMI_CLIMATE_SL_IBI_area_averaged_anomalies/description) 
   
    e. [Mediterranean sea](https://data.marine.copernicus.eu/product/OMI_CLIMATE_SL_MEDSEA_area_averaged_anomalies/description)
### Gridded product
1. [Global ocean mean sea level trend](https://data.marine.copernicus.eu/product/OMI_CLIMATE_SL_GLOBAL_regional_trends/description)

# Import Libraries

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from pathlib import Path

# Function Definition

In [ ]:
def decimal_date(time):
    year_start = pd.Timestamp(year=time.year, month=1, day=1)
    year_end = year_start + pd.DateOffset(years=1)
    decimal_year = time.year + ((time - year_start) / (year_end - year_start))
    return decimal_year

def compute_stats(ds):
    # convert time in decimal year
    t = np.array([decimal_date(pd.to_datetime(ds.time.values[i])) for i in range(len(ds.time.values))])
    # multiply by 10 to convert from cm to mm
    h = ds.MSL_filtered_GIA_corrected_adjusted.values*10
    lingeress = np.polyfit(t-np.mean(t), h, deg=1)
    polyfit= np.polyfit(t-np.mean(t), h, deg=2) # why I am removing mean(t) ? what do you think?
    
    return {
        'trend': lingeress[0],
        'acceleration': polyfit[0]*2,
        'fit': polyfit,
        'unit':{
            'trend': 'mm/yr',
            'acceleration': 'mm^2/yr'
        } 
    }
def plot_result(ds, result_dict, ax=None, color=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(10,5))
        ax.set_xlabel('Time')
        ax.set_ylabel('Sea level anomaly (cm)')

    t = np.array([decimal_date(pd.to_datetime(ds.time.values[i])) for i in range(len(ds.time.values))])
    h = ds.MSL_filtered_GIA_corrected_adjusted.values
    p = np.poly1d(result_dict['fit'])
    ax.plot(t, h, color=color,
            label=f"Trend: {round(result_dict['trend'],2)} mm/yr \nAcceleration: {round(result_dict['acceleration'],2)} $mm^2$/yr")
    ax.plot(t, p(t-np.mean(t))/10, ls='--', color=color)

    return ax

# Time series data

In [ ]:
# Some dictionaries containing file paths and miscellaneous
data_path = Path(r"C:\Users\nyeasm01\OneDrive_univlr_PhD\PhD\Mission\BD_2026\data")
fnames = {
    'msl':{
        'global': data_path/ "omi_climate_sl_global_area_averaged_anomalies_19990220_P20260615.nc",
        'baltic': data_path/ "omi_climate_sl_baltic_area_averaged_anomalies_19990220_P20260615.nc",
        'black_sea': data_path/ "omi_climate_sl_blksea_area_averaged_anomalies_19990220_P20260616.nc",
        'europe': data_path/ "omi_climate_sl_europe_area_averaged_anomalies_19990220_P20260616.nc",
        'iberian_biscay': data_path/ "omi_climate_sl_ibi_area_averaged_anomalies_19990220_P20260616.nc",
        'mediterranean': data_path/ "omi_climate_sl_medsea_area_averaged_anomalies_19990220_P20260616.nc"
    },
    'msl_trend':{
        'global': data_path/ "omi_climate_sl_global_regional_trends_19990220_P20260616.nc"
    }
}
plot_colors = {
    'global': 'red',
    'baltic': 'indigo',
    'black_sea': 'teal',
    'europe': 'blue',
    'iberian_biscay':'orchid',
    'mediterranean':'darkorange'
}

In [ ]:
# Loading the data
ds_gmsl = xr.open_dataset(fnames['msl']['global'])
ds_gmsl_trend = xr.open_dataset(fnames['msl_trend']['global'])
ds_msl_balt = xr.open_dataset(fnames['msl']['baltic'])
ds_msl_black = xr.open_dataset(fnames['msl']['black_sea'])
ds_msl_eur = xr.open_dataset(fnames['msl']['europe'])
ds_msl_ibe = xr.open_dataset(fnames['msl']['iberian_biscay'])
ds_msl_med = xr.open_dataset(fnames['msl']['mediterranean'])

In [ ]:
# look what's in the data, what are the variables, what are the unit of the variables etc.
ds_gmsl

## Data visualization

In [ ]:
# Plotting the data
fig, ax = plt.subplots(figsize=(10,5))
for fname in fnames['msl'].keys():
    ds = xr.open_dataset(fnames['msl'][fname])
    if fname == 'global':
        ax.plot(ds.time.values, ds.MSL_filtered_GIA_corrected_adjusted.values, c=plot_colors[fname], zorder=10, label=fname, lw=1.5)
    else:
        ax.plot(ds.time.values, ds.MSL_filtered_GIA_corrected_adjusted.values, c=plot_colors[fname], label=fname, lw=1)

    ds.close()

ax.set_xlabel('Time')
ax.set_ylabel('Sea level anomaly (cm)')
plt.legend()
plt.show()

## Exercise

1. Compute trend and acceleration of global mean sea level anomaly time series.
2. Compute trend and acceleration of regional mean sea level anomaly time series.
3. Compare global and regional results.
4. Find the time when the mean sea level trend exceeded 3 mm/yr.

__1. Trend and acceleration of global mean sea level time series__

Equation to fit 2nd order polynomial
$$ h = a + b t + c t^2 $$

How we can obtain the acceleration?
$$ \frac{d^2 h}{dt^2} = 2 c $$

__1. Global mean sea level trend__

In [ ]:
fig, ax = plt.subplots(figsize=(10,5))
result = compute_stats(ds_gmsl)
ax = plot_result(ds=ds_gmsl, result_dict=result, ax=ax,  color=plot_colors['global'])
ax.legend()
ax.set_xlabel('Time')
ax.set_ylabel('Sea level anomaly (cm)')

__2 & 3 . Regional mean sea level rise trend and acceleration and comparison with global__

In [ ]:
# %matplotlib widget
fig, axes = plt.subplots(figsize=(10,6), nrows=2, ncols=3, sharex=True, constrained_layout=True)
for fname,ax in zip(fnames['msl'].keys(), axes.flatten()):
    ds = xr.open_dataset(fnames['msl'][fname])
    result = compute_stats(ds)
    ax.set_title(fname)
    ax = plot_result(ds=ds, result_dict=result, ax=ax,  color=plot_colors[fname])
    ds.close()
    ax.legend()

for ax in axes[1,:]:
    ax.set_xlabel('Time')

for ax in axes[:,0]:
    ax.set_ylabel('Sea level anomaly (cm)')

__4. Time when msl trend exceeded 3 mm/yr__

Remember the equation for 2nd order polynomial fit,
$$ h = a + b t + c t^2 $$

So, the slope of this equation is,
$$ \frac{dh}{dt} = b + 2 ct $$

In [ ]:
t = np.array([decimal_date(pd.to_datetime(ds_gmsl.time.values[i])) for i in range(len(ds_gmsl.time.values))])
# x = ds_gmsl.time.values.astype(float)
x = t - np.mean(t)
h = ds_gmsl.MSL_filtered_GIA_corrected_adjusted.values*10
polyfit, cov_p = np.polyfit(x, h, deg=2,cov=True)

p = np.poly1d(polyfit)
plt.figure()
plt.plot(t, polyfit[1]+2*polyfit[0]*x)
plt.axvline(t[np.where(polyfit[1]+2*polyfit[0]*x>=3)[0][0]], c='red', lw=2, ls='--')
plt.xlabel('Time')
plt.ylabel('Trend (mm/yr)')
plt.show()

# Gridded data
## Data visualisation

__Regional mean sea level trend and acceleration:__

In [ ]:
fig, ax = plt.subplots(figsize=(12,6), subplot_kw = {'projection': ccrs.PlateCarree()})
ds_gmsl_trend.trend_GIA_corrected.plot(x='longitude',y='latitude', ax=ax,
                                           cmap='Spectral_r', vmin=-10, vmax=10,
                                           cbar_kwargs={
                                               'label': 'Regional sea level trend (mm/year)',
                                               'shrink': 0.8
                                           })
ax.add_feature(cfeature.LAND, facecolor='dimgray', edgecolor='none',linewidth=0.5)
gl = ax.gridlines(
    crs=ccrs.PlateCarree(), 
    draw_labels=True,             
    linewidth=0.75, color='gray', 
    alpha=0.6, linestyle='--',
    zorder=2
)
gl.right_labels = False
gl.top_labels = False

In [ ]:
fig, ax = plt.subplots(figsize=(12,6), subplot_kw = {'projection': ccrs.PlateCarree()})
ds_gmsl_trend.acceleration.plot(x='longitude',y='latitude', 
                                           cmap='seismic', vmin=-2, vmax=2,
                                           cbar_kwargs={
                                               'label': 'Regional sea level acceleration (mm/year^2)',
                                               'shrink': 0.8
                                           })
ax.add_feature(cfeature.LAND, facecolor='dimgray', edgecolor='none',linewidth=0.5)
gl = ax.gridlines(
    crs=ccrs.PlateCarree(), 
    draw_labels=True,             
    linewidth=0.75, color='gray', 
    alpha=0.6, linestyle='--',
    zorder=2
)
gl.right_labels = False
gl.top_labels = False

## What we can observe?